# Action Generation

## openai api call

In [ ]:
api_key = ""


headers = {
  "Content-Type": "application/json",
  "Authorization": f"Bearer {api_key}"
}


for scenario in scenarios:
  print("---"+scenario+"---"+"\n")
  payload = {
      "model": "gpt-4o",
      "messages": [

        {
          "role": "user",
          "content": [

            {
              "type": "text",
              "text": f"You need to generate different actions in a scenario based on those ten different human values. Your answer should focus only on the action itself and not include other aspects like benefits. The action for each human value should be no more than two sentences. The scenario is: {scenario}. The ten human values are: {human_values} Describe diiferent actions you would take in this scenario. The format of your output should be like: {{\"human_value\": \"action\"}}"
            }

          ]
        }
      ],
      "max_tokens": 4096,
      "temperature": 0
    }

  response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)
  text = response.json()["choices"][0]["message"]["content"]
  print(text+"\n\n")

  with open("4o_actions_reddit.txt","a+") as file:
    file.write("----"+scenario+"----"+"\n")
    file.write(text+"\n\n")



## qwen api call

In [ ]:
from together import Together
import os
os.environ["TOGETHER_API_KEY"] = ""

client = Together()
for scenario in scenarios:
  print("---"+scenario+"---"+"\n")
  response = client.chat.completions.create(
      model="Qwen/Qwen2.5-72B-Instruct-Turbo",
      messages=[{"role": "user", "content": f"You need to generate different actions in a scenario based on those ten different human values. Your answer should focus only on the action itself and not include other aspects like benefits. The action for each human value should be no more than two sentences. The scenario is: {scenario}. The ten human values are: {human_values} Describe diiferent actions you would take in this scenario. The format of your output should be like: {{\"human_value\": \"action\"}}"}],
  )
  text=response.choices[0].message.content
  print(text+"\n\n")
  with open("qwen_actions_reddit.txt","a+") as file:
    file.write("----"+scenario+"----"+"\n")
    file.write(text+"\n\n")


##Merge


In [ ]:
import json
import ast
import re
import pandas as pd
import random

def parse_model_qwen(text):
    all_scenes = []

    pattern = r"-{4}.*?-{4}\s*((?:\{.*?\}\s*){10})"
    matches = re.findall(pattern, text, re.DOTALL)
    print(len(matches))
    for block in matches:
      lines = block.strip().split("\n")
      try:
        scene = [json.loads(line.strip()) for line in lines if line.strip()]
        merged_scene = {}
        for d in scene:
            merged_scene.update(d)
      except:
        print(lines)
        continue


      all_scenes.append(merged_scene)
    print(len(all_scenes))

    return all_scenes



def parse_model_gpt(text):
    all_scenes = []

    scenario_blocks = list(re.finditer(r"-{4}(.*?)-{4}(.*?)(?=-{4}|$)", text, re.DOTALL))

    for i,match in enumerate(scenario_blocks):

        content = match.group(2).strip()
        extracted = None

        # Try format 1: ```json { big dictionary } ```
        codeblock_match = re.search(r"```json\s*(\{[\s\S]*?\})\s*```", content)
        if codeblock_match:
            try:
                data = json.loads(codeblock_match.group(1))
                if isinstance(data, dict) and len(data) == 10:
                    extracted = data
            except json.JSONDecodeError:
                pass

        # Try format 2: unwrapped single big dictionary
        if extracted is None:
            unwrapped_dict_match = re.search(r"^\s*(\{[\s\S]*?\})\s*$", content)
            if unwrapped_dict_match:
                try:
                    data = json.loads(unwrapped_dict_match.group(1))
                    if isinstance(data, dict) and len(data) == 10:
                        extracted = data
                except json.JSONDecodeError:
                    pass

        # Try format 3: ten separate JSON lines
        if extracted is None:
            try:
                lines = content.strip().split("\n")
                if len(lines) >= 10:
                    items = [json.loads(line.strip()) for line in lines if line.strip()]
                    if len(items) == 10:
                        scene = {}
                        for d in items:
                            scene.update(d)
                        if len(scene) == 10:
                            extracted = scene
            except json.JSONDecodeError:
               print(content)
        all_scenes.append(extracted)

    print(len(all_scenes))
    return all_scenes



with open("qwen_actions_reddit.txt", "r", encoding="utf-8") as f1:
    qwen_text = f1.read()
    qwen_scenes = parse_model_qwen(qwen_text)

with open("4o_actions_reddit.txt", "r", encoding="utf-8") as f2:
    gpt_text = f2.read()
    gpt_scenes = parse_model_gpt(gpt_text)


df = pd.DataFrame({
    'qwen': [json.dumps(scene, ensure_ascii=False) for scene in qwen_scenes],
    'gpt': [json.dumps(scene, ensure_ascii=False) for scene in gpt_scenes]
})

scenarios = scenarios.reset_index(drop=True)
df = df.reset_index(drop=True)

merged_df = pd.concat([scenarios,df], axis=1)

merged_df.to_csv('scenario_action.csv', index=False)

qwen_actions = merged_df.iloc[:, 1].apply(json.loads)
gpt_actions = merged_df.iloc[:, 2].apply(json.loads)



In [ ]:
import random

def build_pair_pool(qwen_scene, gpt_scene):
    pool = []
    for value in qwen_scene:
        pool.append(("qwen", value, qwen_scene[value]))
        pool.append(("gpt-4o", value, gpt_scene[value]))
    return pool

import random

def sample_five_rounds_with_retry(qwen_scene, gpt_scene, max_retry=1000):
    """
    Returns:
        [
          [ (model, value, action), (model, value, action), (model, value, action), (model, value, action) ],
          ... (5 rounds total)
        ]
    """

    for _ in range(max_retry):
        try:
            # Step 1: build 20 candidates
            remaining = []
            for value in qwen_scene:
                remaining.append(("qwen", value, qwen_scene[value]))
                remaining.append(("gpt-4o", value, gpt_scene[value]))

            all_rounds = []

            # Step 2: sample 5 rounds
            for _ in range(5):
                round_pairs = []
                used_values = set()

                candidates = remaining.copy()
                random.shuffle(candidates)

                for pair in candidates:
                    model, value, action = pair

                    # value cannot repeat within a round
                    if value in used_values:
                        continue

                    round_pairs.append(pair)
                    used_values.add(value)
                    remaining.remove(pair)

                    if len(round_pairs) == 4:
                        break

                if len(round_pairs) < 4:
                    raise RuntimeError("round failed")

                all_rounds.append(round_pairs)

            # Step 3: ensure all 20 are consumed
            if len(remaining) != 0:
                raise RuntimeError("not fully consumed")

            return all_rounds

        except RuntimeError:
            continue

    raise RuntimeError("Sampling failed after max_retry attempts")




for i, scenario in enumerate(scenarios):
    qwen_scene = qwen_actions[i]
    gpt_scene = gpt_actions[i]

    rounds = sample_five_rounds_with_retry(qwen_scene, gpt_scene)

    with open("qa_pairs_reddit.txt", "a+") as f:
        f.write(str(rounds) + "\n")
